###E5

In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1378")
catalog = dbutils.widgets.get("catalog")

In [0]:
from decimal import Decimal
from pyspark.sql import Row, functions as F

# E5: ops.reconciliation_results — append 5 reconciliation checks per pipeline run
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts        TIMESTAMP,
    check_name    STRING,
    source_value  DECIMAL(18,2),
    target_value  DECIMAL(18,2),
    status        STRING
)
""")

# Helper: count a table safely (return None if missing)
def safe_count(table_name):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None

# 1. ORDER_COUNT — distinct standardised order_id in bronze.orders vs silver.orders + quarantine_orders
order_count_src = spark.table(f"{catalog}.bronze.orders") \
    .agg(F.countDistinct(F.upper(F.trim("order_id")))).collect()[0][0]
order_count_tgt = spark.table(f"{catalog}.silver.orders").count() \
    + spark.table(f"{catalog}.silver.quarantine_orders").count()

# 2. LINE_COUNT — rows in silver.order_items vs rows in gold.fact_order_line
line_count_src = spark.table(f"{catalog}.silver.order_items").count()
line_count_tgt = safe_count(f"{catalog}.gold.fact_order_line")

# 3. NET_REVENUE — sum(net_amount) for DELIVERED orders in silver.order_items vs sum(net_item_revenue) in agg_daily_store_sales
net_rev_src = spark.sql(f"""
    SELECT COALESCE(SUM(oi.net_amount), 0)
    FROM {catalog}.silver.order_items oi
    JOIN {catalog}.silver.orders so
      ON oi.order_id = so.order_id AND oi.updated_at = so.updated_at
    WHERE so.order_status = 'DELIVERED'
""").collect()[0][0]
net_rev_tgt = spark.table(f"{catalog}.gold.agg_daily_store_sales") \
    .agg(F.sum("net_item_revenue")).collect()[0][0]

# 4. DELIVERED_ORDERS — DELIVERED rows in gold.fact_orders vs sum(delivered_orders) in agg_daily_store_sales
delivered_src = spark.table(f"{catalog}.gold.fact_orders").filter("order_status = 'DELIVERED'").count()
delivered_tgt = spark.table(f"{catalog}.gold.agg_daily_store_sales") \
    .agg(F.sum("delivered_orders")).collect()[0][0]

# 5. CURRENT_CUSTOMERS — distinct customer_id in silver.customers vs rows in dim_customer where is_current = TRUE
cust_src = spark.table(f"{catalog}.silver.customers").agg(F.countDistinct("customer_id")).collect()[0][0]
cust_tgt = spark.table(f"{catalog}.gold.dim_customer").filter("is_current = TRUE").agg(F.countDistinct("customer_id")).collect()[0][0]

# Clear previous run results (keep table schema, fresh append for this run)
spark.sql(f"TRUNCATE TABLE {catalog}.ops.reconciliation_results")

# Build 5 rows and append
def to_dec(val):
    return Decimal(str(val)) if val is not None else None

def check_status(src, tgt):
    if src is None or tgt is None:
        return "FAIL"
    return "PASS" if src == tgt else "FAIL"

checks = [
    ("ORDER_COUNT",       to_dec(order_count_src), to_dec(order_count_tgt)),
    ("LINE_COUNT",         to_dec(line_count_src),  to_dec(line_count_tgt)),
    ("NET_REVENUE",        to_dec(net_rev_src),     to_dec(net_rev_tgt)),
    ("DELIVERED_ORDERS",   to_dec(delivered_src),   to_dec(delivered_tgt)),
    ("CURRENT_CUSTOMERS",  to_dec(cust_src),        to_dec(cust_tgt)),
]

rows = [Row(check_name=c[0], source_value=c[1], target_value=c[2],
            status=check_status(c[1], c[2])) for c in checks]

results = spark.createDataFrame(rows) \
    .withColumn("source_value", F.col("source_value").cast("decimal(18,2)")) \
    .withColumn("target_value", F.col("target_value").cast("decimal(18,2)")) \
    .withColumn("run_ts", F.current_timestamp()) \
    .select("run_ts", "check_name", "source_value", "target_value", "status")

results.write.mode("append").saveAsTable(f"{catalog}.ops.reconciliation_results")

# Verify
print(f"Appended {results.count()} reconciliation rows")
display(spark.table(f"{catalog}.ops.reconciliation_results"))

In [0]:
# H2: ops.checkpoint_counts — append checkpoint info after each run
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_counts (
    checkpoint_name STRING,
    table_name STRING,
    row_count BIGINT,
    captured_at TIMESTAMP
)
""")

# List tables for checkpoints
checkpoint_tables = [
    "bronze.orders", "bronze.customers", "bronze.products", "bronze.stores",
    "silver.orders", "silver.order_items", "silver.quarantine_orders", "silver.customers", "silver.products",
    "gold.dim_customer", "gold.fact_orders", "gold.fact_order_line", "gold.agg_daily_store_sales"
]

# Function to append checkpoint count
from pyspark.sql import Row
from datetime import datetime

def append_checkpoint_counts(checkpoint_name):
    rows = []
    for tbl in checkpoint_tables:
        try:
            cnt = spark.table(f"{catalog}.{tbl}").count()
        except Exception:
            cnt = None
        rows.append(Row(
            checkpoint_name=checkpoint_name,
            table_name=tbl,
            row_count=cnt,
            captured_at=datetime.now()
        ))
    df = spark.createDataFrame(rows).select("checkpoint_name", "table_name", "row_count", "captured_at")
    df.write.mode("append").saveAsTable(f"{catalog}.ops.checkpoint_counts")

# Example usage:
# append_checkpoint_counts("AFTER_BATCH_1")
# append_checkpoint_counts("AFTER_BATCH_2")
# append_checkpoint_counts("AFTER_RERUN")

# H3: ops.load_audit — audit job/task writes
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.load_audit (
    job_run_id STRING,
    task_name STRING,
    target_table STRING,
    row_count BIGINT,
    completed_at TIMESTAMP
)
""")

# Function to audit a load

def append_load_audit(job_run_id, task_name, target_tables):
    rows = []
    for tbl in target_tables:
        try:
            cnt = spark.table(f"{catalog}.{tbl}").count()
        except Exception:
            cnt = None
        rows.append(Row(
            job_run_id=job_run_id,
            task_name=task_name,
            target_table=tbl,
            row_count=cnt,
            completed_at=datetime.now()
        ))
    df = spark.createDataFrame(rows).select("job_run_id", "task_name", "target_table", "row_count", "completed_at")
    df.write.mode("append").saveAsTable(f"{catalog}.ops.load_audit")

# Example usage:
# append_load_audit("run_001", "batch_1_task", ["bronze.orders", "bronze.customers"])
# append_load_audit("run_002", "batch_2_task", ["silver.orders"])


In [0]:
%sql
select * from fk_ce_2026_07_1378.ops.checkpoint_counts